# 14 · Machine learning: classification with rare events
Goal: predict which records end up **Delayed or Cancelled**. About 1 in 14 do. This is where "accuracy" misleads, and where precision, recall and thresholds matter.

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import numpy as np
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv
df["not_completed"] = (df["status"] != "Completed").astype(int)
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
NUM, CAT = ["units_processed", "duration_minutes"], ["category"]
FEATURES = NUM + CAT
days = np.sort(df["record_date"].unique())
cutoff = days[int(len(days) * 0.8)]
train, test = df[df["record_date"] < cutoff], df[df["record_date"] >= cutoff]
y_train, y_test = train["not_completed"], test["not_completed"]
print(f"problem rate: train {y_train.mean():.1%}, test {y_test.mean():.1%}")

def make(**kw):
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000, **kw))])

## The accuracy trap
A model that **always says "Completed"** is right about 93% of the time and catches zero problems.

In [ ]:
always_ok = np.zeros(len(y_test), dtype=int)
print(f"always-'Completed' accuracy: {accuracy_score(y_test, always_ok):.1%}   recall: {recall_score(y_test, always_ok, zero_division=0):.0%}")
pipe = make().fit(train[FEATURES], y_train)
proba = pipe.predict_proba(test[FEATURES])[:, 1]
def report(threshold):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {"threshold": threshold, "accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0), "f1": f1_score(y_test, pred, zero_division=0), "caught": tp, "false_alarms": fp, "missed": fn}
pd.DataFrame([report(t) for t in (0.5, 0.3, 0.15, 0.1, 0.07)]).round(3)

**Reading it:** precision = of the records we flagged, how many were truly problems. Recall = of the real problems, how many we flagged. Lowering the threshold catches more (recall up) but raises false alarms (precision down). The right point depends on the **cost of each mistake**, which is a business question.

## Your turn 1: balance the classes
Retrain with `class_weight="balanced"` (it up-weights the rare class) using `make(class_weight="balanced")`. Store the recall at threshold 0.5 in `balanced_recall`.

In [ ]:
balanced_recall = None

In [ ]:
base_recall = recall_score(y_test, (proba >= 0.5).astype(int), zero_division=0)
check("balanced model has higher recall than the plain one", balanced_recall is not None and balanced_recall > base_recall,
      "p = make(class_weight='balanced').fit(...).predict_proba(...)[:, 1]; recall_score(y_test, p >= 0.5)")

## Your turn 2: choose a threshold the right way
Picking the threshold by looking at the *test* set is cheating (you'd be tuning to the exam). Instead: split the **training** period again, fit on the earlier part, find the best threshold on the later part (validation), then report on test once.

`val_y` and `val_proba` are given. Try thresholds `np.arange(0.02, 0.5, 0.02)`, and store the one with the highest F1 on validation in `best_threshold` (on ties, the smallest).

In [ ]:
tdays = np.sort(train["record_date"].unique())
vcut = tdays[int(len(tdays) * 0.75)]
fit_part, val_part = train[train["record_date"] < vcut], train[train["record_date"] >= vcut]
val_model = make().fit(fit_part[FEATURES], fit_part["not_completed"])
val_y = val_part["not_completed"].to_numpy()
val_proba = val_model.predict_proba(val_part[FEATURES])[:, 1]
best_threshold = None

In [ ]:
thresholds = np.arange(0.02, 0.5, 0.02)
truth = thresholds[int(np.argmax([f1_score(val_y, (val_proba >= t).astype(int), zero_division=0) for t in thresholds]))]
check("best_threshold", best_threshold is not None and abs(best_threshold - truth) < 1e-9, "loop over thresholds, f1_score(val_y, val_proba >= t), take the argmax")
if best_threshold is not None:
    final = report(best_threshold)
    print(f"on test at your threshold: precision {final['precision']:.2f}, recall {final['recall']:.2f}, F1 {final['f1']:.2f}  (vs F1 {report(0.5)['f1']:.2f} at 0.5)")

**Discuss:** with rare events, also look at a precision-recall curve rather than ROC, and report results with their uncertainty: the test set contains only a few dozen real problems.